# Classification with a Multi-Layer Perceptron (MLP)

- Introducing Softmax and Cross-Entropy Loss Function and a Non-Linear Activation Function (ReLU)
Ideal for a probabilistic distribution output and a non-linear decision boundary.

Softmax: Converts raw output scores (logits) into probabilities that sum to 1.
- Y = Softmax(X) = exp(X) / sum(exp(X)) 

Cross-Entropy Loss: Measures the difference between the predicted probability distribution and the true distribution (one-hot encoded labels).
- L = -sum(y_true * log(y_pred))
- dL/dX = y_pred - y_true (when using softmax + cross-entropy together, this simplifies the gradient calculation)


` Note: CrossEntropy applied outside of the model, because of calc optimization,
as consequent exec of exp then log may lead to numerical instability. 
If applied outside of the model, we can use log(exp(...)) optimization to calc 
forward+backward pass alltogether, which is more stable.`

In [ ]:
pip install scikit-learn

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Pipeline: Data Preparation
# Iris is a classic dataset with 150 samples, 4 features, and 3 classes 
# representing different iris species.
iris = load_iris()
X, y = iris.data, iris.target

# Split into Training (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalization (The "Outside" way, though we could bake it in)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train) # Long is required for CrossEntropy labels
y_test = torch.LongTensor(y_test)


In [12]:
# 2. Pipeline: Architecture
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.layer1 = nn.Linear(4, 10) # 4 inputs -> 10 hidden
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(10, 3) # 10 hidden -> 3 classes
        # Note: CrossEntropyLoss in PyTorch applies Softmax internally!

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

In [13]:

# 3. Pipeline: Training Logic
model = IrisNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) # Adam is smarter than SGD

In [14]:

for epoch in range(100):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [15]:

# 4. Pipeline: Evaluation
model.eval() # Set to evaluation mode
with torch.no_grad():
    test_outputs = model(X_test)
    # Get index of max probability
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 100.00%
